# Optimizador y asignador de proveedores

Versión preparada para integrarse posteriormente con una aplicación web.

**Entradas de la consulta:** código de tratamiento, código de municipio y umbral mínimo de valoración.

**Capacidad:** se utiliza exclusivamente `CAPACIDAD_ANUAL` del archivo de capacidad. `CAPACIDAD_CONSUMIDA` se ignora completamente. El consumo comienza en 0 y aumenta en 1 por cada paciente asignado durante la ejecución.

**Ranking:** coste (50 %), valoración (30 %) y capacidad disponible (20 %).

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

RUTA_EXCEL = Path("datos_tratamientos_detalle_con_rating_capacidad.xlsx")
TOP_N = 10
PESO_COSTE = 0.50
PESO_VALORACION = 0.30
PESO_CAPACIDAD = 0.20

df = pd.read_excel(RUTA_EXCEL)
print(df.shape)
print(df.columns.tolist())

(72391, 16)
['IDEPREADMIN', 'NUMLIQUID', 'CODIGO', 'DESCPROCED', 'TIPOIDBENEFSIN', 'NUMIDBENEFSIN', 'DVIDBENEFSIN', 'CODPROVEEDOR', 'NOMBREPROVEEDOR', 'TIPOPREADMIN', 'DESCRIPTIPOPRE', 'CODIGO_MUNICIPIO', 'DESCMUNICIPIO', 'SUMA', 'rating_global', 'CAPACIDAD_ANUAL']


In [ ]:
# Preparación
columnas = [
    "CODIGO","DESCPROCED","CODPROVEEDOR","NOMBREPROVEEDOR",
    "CODIGO_MUNICIPIO","DESCMUNICIPIO","SUMA","rating_global",
    "CAPACIDAD_ANUAL"
]
faltantes=[c for c in columnas if c not in df.columns]
if faltantes:
    raise KeyError(f"Faltan columnas: {faltantes}")

datos=df[columnas].copy()
for c in ["CODIGO","CODPROVEEDOR","SUMA","rating_global","CAPACIDAD_ANUAL"]:
    datos[c]=pd.to_numeric(datos[c],errors="coerce")
datos["CODIGO_MUNICIPIO"]=datos["CODIGO_MUNICIPIO"].astype(str).str.strip()
datos["DESCMUNICIPIO"]=datos["DESCMUNICIPIO"].astype(str).str.strip()
datos["rating_global"]=datos["rating_global"].fillna(3.0)

In [ ]:
# ============================================================
# CREAR ESTADO DE CAPACIDAD Y TABLA PARA EL RANKING
# ============================================================

# 1. Tabla de capacidad por proveedor + tratamiento
capacidades = (
    datos[
        [
            "CODPROVEEDOR",
            "CODIGO",
            "CAPACIDAD_ANUAL"
        ]
    ]
    .dropna(
        subset=[
            "CODPROVEEDOR",
            "CODIGO",
            "CAPACIDAD_ANUAL"
        ]
    )
    .drop_duplicates(
        subset=[
            "CODPROVEEDOR",
            "CODIGO"
        ]
    )
    .copy()
)

capacidades["CAPACIDAD_ANUAL"] = (
    pd.to_numeric(
        capacidades["CAPACIDAD_ANUAL"],
        errors="coerce"
    )
    .fillna(0)
    .astype(int)
)

# 2. El programa empieza con 0 pacientes asignados
capacidades["ASIGNADOS"] = 0

# 3. Capacidad disponible inicialmente
capacidades["CAPACIDAD_RESTANTE"] = (
    capacidades["CAPACIDAD_ANUAL"]
    - capacidades["ASIGNADOS"]
)

# No puede existir capacidad negativa
capacidades["CAPACIDAD_RESTANTE"] = (
    capacidades["CAPACIDAD_RESTANTE"]
    .clip(lower=0)
)

print(
    f"Combinaciones proveedor-tratamiento con capacidad: "
    f"{len(capacidades):,}"
)

# ============================================================
# 4. Incorporar el estado de capacidad a los datos
# ============================================================

datos = datos.merge(
    capacidades[
        [
            "CODPROVEEDOR",
            "CODIGO",
            "ASIGNADOS",
            "CAPACIDAD_RESTANTE"
        ]
    ],
    on=[
        "CODPROVEEDOR",
        "CODIGO"
    ],
    how="left"
)

# ============================================================
# 5. Crear una opción única para el ranking
# ============================================================

opciones_base = (
    datos
    .dropna(
        subset=[
            "CAPACIDAD_ANUAL",
            "ASIGNADOS",
            "CAPACIDAD_RESTANTE"
        ]
    )
    .groupby(
        [
            "CODPROVEEDOR",
            "NOMBREPROVEEDOR",
            "CODIGO",
            "DESCPROCED",
            "CODIGO_MUNICIPIO",
            "DESCMUNICIPIO"
        ],
        as_index=False
    )
    .agg(
        Coste=("SUMA", "mean"),
        Valoracion=("rating_global", "mean"),
        CapacidadAnual=("CAPACIDAD_ANUAL", "first"),
        Asignados=("ASIGNADOS", "first"),
        CapacidadRestante=("CAPACIDAD_RESTANTE", "first")
    )
)

print(f"Opciones únicas para el ranking: {len(opciones_base):,}")

display(opciones_base.head())

Combinaciones proveedor-tratamiento con capacidad: 9,253
Opciones únicas para el ranking: 9,261


,CODPROVEEDOR,NOMBREPROVEEDOR,CODIGO,DESCPROCED,CODIGO_MUNICIPIO,DESCMUNICIPIO,Coste,Valoracion,CapacidadAnual,Asignados,CapacidadRestante
0,5,CENTRO MEDICO LOIRA C.A,109020009,EVALUACION CARDIOVASCULAR PREOPERATORIA CARDIO...,001-002-004,LIBERTADOR,100.0,4.2,20.0,0.0,20.0
1,5,CENTRO MEDICO LOIRA C.A,120010009,ANESTESIOLOGIA VALORACION PREANESTESICA,001-002-004,LIBERTADOR,40.0,4.2,20.0,0.0,20.0
2,5,CENTRO MEDICO LOIRA C.A,401010127,GRUPO SANGUINEO ABO Y FACTOR RH,001-002-004,LIBERTADOR,80.0,4.2,20.0,0.0,20.0
3,5,CENTRO MEDICO LOIRA C.A,401200051,LABORATORIO GENERAL (SOLO PARA CARTA AVAL / RE...,001-002-004,LIBERTADOR,150.0,4.2,20.0,0.0,20.0
4,5,CENTRO MEDICO LOIRA C.A,402020068,COLANGIORESONANCIA,001-002-004,LIBERTADOR,280.0,4.2,20.0,0.0,20.0


In [ ]:
def _beneficio(s):
    if s.max()==s.min(): return pd.Series(1.0,index=s.index)
    return (s-s.min())/(s.max()-s.min())

def _coste(s):
    if s.max()==s.min(): return pd.Series(1.0,index=s.index)
    return (s.max()-s)/(s.max()-s.min())

def calcular_ranking(candidatos):
    x=candidatos.copy()
    x["PctCapacidadDisponible"]=x["CapacidadRestante"]/x["CapacidadAnual"]
    x["ScoreCoste"]=_coste(x["Coste"])
    x["ScoreValoracion"]=_beneficio(x["Valoracion"])
    x["ScoreCapacidad"]=_beneficio(x["PctCapacidadDisponible"])
    x["IndiceRanking"]=(PESO_COSTE*x["ScoreCoste"]+
                        PESO_VALORACION*x["ScoreValoracion"]+
                        PESO_CAPACIDAD*x["ScoreCapacidad"])
    x=x.sort_values(["IndiceRanking","Coste","Valoracion"],
                    ascending=[False,True,False]).reset_index(drop=True)
    x["Ranking"]=np.arange(1,len(x)+1)
    return x

In [ ]:
def obtener_ranking(tratamiento, municipio, rating_minimo=0.0, top_n=TOP_N):
    candidatos=opciones_base[
        (opciones_base["CODIGO"]==tratamiento) &
        (opciones_base["CODIGO_MUNICIPIO"].astype(str)==str(municipio).strip()) &
        (opciones_base["Valoracion"]>=rating_minimo) &
        (opciones_base["CapacidadRestante"]>0)
    ].copy()
    if candidatos.empty:
        return candidatos
    return calcular_ranking(candidatos).head(top_n)

# Ejemplo automático
ej=opciones_base.iloc[0]
ranking=obtener_ranking(int(ej.CODIGO),ej.CODIGO_MUNICIPIO,3.0)
ranking

,CODPROVEEDOR,NOMBREPROVEEDOR,CODIGO,DESCPROCED,CODIGO_MUNICIPIO,DESCMUNICIPIO,Coste,Valoracion,CapacidadAnual,Asignados,CapacidadRestante,PctCapacidadDisponible,ScoreCoste,ScoreValoracion,ScoreCapacidad,IndiceRanking,Ranking
0,304,VISION PARAISOC.A,109020009,EVALUACION CARDIOVASCULAR PREOPERATORIA CARDIO...,001-002-004,LIBERTADOR,50.000000,3.9,20.0,0.0,20.0,1.0,1.000000,0.363636,1.0,0.809091,1
1,10,FUNDACION HOSPITAL ORTOPEDICO INFANTIL,109020009,EVALUACION CARDIOVASCULAR PREOPERATORIA CARDIO...,001-002-004,LIBERTADOR,80.000000,4.6,20.0,0.0,20.0,1.0,0.400000,1.000000,1.0,0.700000,2
2,1134,CENTRO CLINICO FENIX SALUD C.A,109020009,EVALUACION CARDIOVASCULAR PREOPERATORIA CARDIO...,001-002-004,LIBERTADOR,71.931406,4.0,70.0,0.0,70.0,1.0,0.561372,0.454545,1.0,0.617050,3
3,254,CLINICA HERRERA LYNCH C.A,109020009,EVALUACION CARDIOVASCULAR PREOPERATORIA CARDIO...,001-002-004,LIBERTADOR,80.000000,4.0,20.0,0.0,20.0,1.0,0.400000,0.454545,1.0,0.536364,4
4,694,INSTITUTO CLINICO LA FLORIDA C.A.,109020009,EVALUACION CARDIOVASCULAR PREOPERATORIA CARDIO...,001-002-004,LIBERTADOR,100.000000,4.5,20.0,0.0,20.0,1.0,0.000000,0.909091,1.0,0.472727,5
5,5,CENTRO MEDICO LOIRA C.A,109020009,EVALUACION CARDIOVASCULAR PREOPERATORIA CARDIO...,001-002-004,LIBERTADOR,100.000000,4.2,20.0,0.0,20.0,1.0,0.000000,0.636364,1.0,0.390909,6
6,140,CLINICA CEMO C.A,109020009,EVALUACION CARDIOVASCULAR PREOPERATORIA CARDIO...,001-002-004,LIBERTADOR,90.000000,3.8,20.0,0.0,20.0,1.0,0.200000,0.272727,1.0,0.381818,7
7,462,CENTRO ORTOPEDICO Y PODOLOGICO,109020009,EVALUACION CARDIOVASCULAR PREOPERATORIA CARDIO...,001-002-004,LIBERTADOR,86.666667,3.5,20.0,0.0,20.0,1.0,0.266667,0.000000,1.0,0.333333,8
8,3009,CLINICA LOS SAUCES C.A.,109020009,EVALUACION CARDIOVASCULAR PREOPERATORIA CARDIO...,001-002-004,LIBERTADOR,100.000000,3.9,20.0,0.0,20.0,1.0,0.000000,0.363636,1.0,0.309091,9


In [ ]:
historial_asignaciones=[]

def asignar_paciente(cod_proveedor, cod_tratamiento):
    global capacidades, opciones_base

    mask=(capacidades.CODPROVEEDOR==cod_proveedor)&(capacidades.CODIGO==cod_tratamiento)
    idxs=capacidades.index[mask]
    if len(idxs)==0:
        return {"ok":False,"mensaje":"No existe capacidad registrada para proveedor-tratamiento."}

    idx=idxs[0]
    anual=int(capacidades.at[idx,"CAPACIDAD_ANUAL"])
    asignados=int(capacidades.at[idx,"ASIGNADOS"])
    restante=anual-asignados

    if restante<=0:
        return {"ok":False,"mensaje":"El proveedor ha alcanzado su capacidad máxima."}

    capacidades.at[idx,"ASIGNADOS"]=asignados+1
    capacidades.at[idx,"CAPACIDAD_RESTANTE"]=anual-asignados-1

    mask2=(opciones_base.CODPROVEEDOR==cod_proveedor)&(opciones_base.CODIGO==cod_tratamiento)
    opciones_base.loc[mask2,"Asignados"]=asignados+1
    opciones_base.loc[mask2,"CapacidadRestante"]=anual-asignados-1

    registro={"CODPROVEEDOR":cod_proveedor,"CODIGO":cod_tratamiento,
              "CapacidadAnual":anual,"Asignados":asignados+1,
              "CapacidadRestante":anual-asignados-1}
    historial_asignaciones.append(registro)

    return {"ok":True,**registro,
            "mensaje":"Proveedor agotado." if anual-asignados-1==0
                     else f"Quedan {anual-asignados-1} plazas."}

## Integración con la aplicación web

La web puede llamar a `obtener_ranking(tratamiento, municipio, rating_minimo, top_n)` para recibir las opciones.

Cuando el usuario selecciona una opción, llama a `asignar_paciente(cod_proveedor, cod_tratamiento)`.

**Importante para producción:** el contador `ASIGNADOS` de este cuaderno está en memoria. Si la web tendrá varios usuarios o procesos simultáneos, las asignaciones deben persistirse en una base de datos y actualizarse de forma transaccional para evitar asignar dos veces la misma última plaza.